# Agentic Email Marketing Service
Built with LangGraph, SendGrid, and Gradio.

### Import relevant libraries

In [ ]:
import os
import json
import io
import asyncio
import gradio as gr
import sendgrid
from typing import List, Optional
from typing_extensions import TypedDict
from pydantic import BaseModel, Field
from duckduckgo_search import DDGS
from sendgrid.helpers.mail import Content, Email, Mail, To
from dotenv import load_dotenv
from rich.console import Console
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.tools import tool
from langgraph.graph import StateGraph, START, END
from langgraph.prebuilt import create_react_agent

load_dotenv(override=True)

console = Console(force_terminal=True, width=80)
_trace_console: Optional[Console] = None


def _active_console() -> Console:
    return _trace_console if _trace_console is not None else console

### Set up environment

In [ ]:
SENDER_EMAIL = "answers@ailysis.io"

OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY")
SENDGRID_API_KEY = os.environ.get("SENDGRID_API_KEY")

print("Environment ready.")

### Business Tools

In [ ]:
def _web_search_impl(query: str) -> str:
    """Search the web for market trends and competitor info."""
    _active_console().print(f"[bold cyan]Researcher is searching for:[/bold cyan] {query}")
    try:
        with DDGS() as ddgs:
            results = [r for r in ddgs.text(query, max_results=3)]
        return json.dumps(results) if results else "No specific results found for this query."
    except Exception as e:
        return f"Search error: {str(e)}"

def _fetch_subscriber_emails_impl() -> List[str]:
    """Registered subscriber emails."""
    emails = ["xxxx@gmail.com", "xxxxx@gmail.com"]
    _active_console().print(
        f"[bold magenta]Database:[/bold magenta] Retrieved {len(emails)} subscribers."
    )
    return emails

def _send_marketing_email_impl(subject: str, html_content: str, recipients: List[str]) -> str:
    """Send via SendGrid."""
    _active_console().print(
        f"[bold green]Dispatching email to {len(recipients)} recipient(s)...[/bold green]"
    )
    api_key = os.environ.get("SENDGRID_API_KEY")
    if not api_key:
        return "Skipped send: SENDGRID_API_KEY is not set (dry run)."
    
    try:
        sg = sendgrid.SendGridAPIClient(api_key=api_key)
        from_email = Email(SENDER_EMAIL)
        for email_addr in recipients:
            to_email = To(email_addr)
            content = Content("text/html", html_content)
            mail = Mail(from_email, to_email, subject, content)
            response = sg.client.mail.send.post(request_body=mail.get())
            if response.status_code not in (200, 201, 202):
                return f"SendGrid error {response.status_code}: {response.body}"
        return f"Successfully sent to {len(recipients)} recipients."
    except Exception as e:
        return f"Error: {str(e)}"

@tool
def web_search(query: str) -> str:
    """Search the web for market trends and competitor info. Query must be a specific search string."""
    return _web_search_impl(query)

def fetch_subscriber_emails() -> List[str]:
    """Queries the database for registered user emails."""
    return _fetch_subscriber_emails_impl()

def send_marketing_email(subject: str, html_content: str, recipients: List[str]) -> str:
    """Sends the finalized marketing email via SendGrid."""
    return _send_marketing_email_impl(subject, html_content, recipients)


### LangGraph pipeline

StateGraph nodes: plan (structured output) → research (parallel create_react_agent + web_search) → write (structured EmailDraft).

In [ ]:
HOW_MANY_SEARCHES = 3
LLM_MODEL = "gpt-4.1-nano"


class WebSearchItem(BaseModel):
    reason: str = Field(description="Why this search matters for the query.")
    query: str = Field(description="The search term to use for the web search.")


class WebSearchPlan(BaseModel):
    searches: List[WebSearchItem] = Field(
        description="Web searches to perform to best answer the marketing research query."
    )


class EmailDraft(BaseModel):
    subject: str = Field(description="Catchy subject line.")
    html_body: str = Field(description="The marketing email in HTML format.")


class PipelineState(TypedDict, total=False):
    """LangGraph state: plan → parallel research (inside research node) → draft."""

    user_prompt: str
    search_plan: Optional[WebSearchPlan]
    search_results: List[str]
    draft: Optional[EmailDraft]
    error: Optional[str]


llm = ChatOpenAI(model=LLM_MODEL, temperature=0)
planner_llm = llm.with_structured_output(WebSearchPlan)
draft_llm = llm.with_structured_output(EmailDraft)

RESEARCHER_SYSTEM = (
    "You are a meticulous market researcher. "
    "Use the web_search tool for up-to-date information, then summarize key findings concisely."
)
research_agent = create_react_agent(
    llm,
    [web_search],
    prompt=SystemMessage(content=RESEARCHER_SYSTEM),
)


def _final_message_text(messages: list) -> str:
    last = messages[-1]
    content = getattr(last, "content", None)
    if isinstance(content, str):
        return content
    if isinstance(content, list):
        parts: List[str] = []
        for block in content:
            if isinstance(block, dict) and block.get("type") == "text":
                parts.append(str(block.get("text", "")))
            elif isinstance(block, str):
                parts.append(block)
        return "".join(parts)
    return str(content or last)


async def plan_searches(query: str) -> Optional[WebSearchPlan]:
    msg = HumanMessage(
        content=(
            f"Given a marketing campaign brief, propose exactly {HOW_MANY_SEARCHES} distinct web searches.\n\n"
            f"Campaign brief:\n{query}"
        )
    )
    return await planner_llm.ainvoke([msg])


async def search_item(item: WebSearchItem) -> str:
    user_msg = HumanMessage(
        content=(
            f"Focus only on this search term: {item.query}\n"
            f"Reason it matters: {item.reason}\n"
            "Call web_search, wait for the result, and summarize key findings for this query."
        )
    )
    result = await research_agent.ainvoke({"messages": [user_msg]})
    return _final_message_text(result["messages"])


async def perform_searches(search_plan: WebSearchPlan) -> List[str]:
    tasks = [asyncio.create_task(search_item(item)) for item in search_plan.searches]
    return list(await asyncio.gather(*tasks))


async def write_email_draft(query: str, search_results: List[str]) -> Optional[EmailDraft]:
    msg = HumanMessage(
        content=(
            "Turn the research into one high-converting HTML marketing email matching the EmailDraft schema.\n\n"
            f"Campaign brief:\n{query}\n\nResearch summaries:\n{search_results}"
        )
    )
    return await draft_llm.ainvoke([msg])


async def node_plan(state: PipelineState) -> dict:
    _active_console().print("[yellow]Planning searches...[/yellow]")
    plan = await plan_searches(state["user_prompt"])
    if not plan or not plan.searches:
        _active_console().print("[bold red]Planning phase yielded no results.[/bold red]")
        return {"search_plan": plan, "error": "empty_plan"}
    _active_console().print(f"[bold green]Planned {len(plan.searches)} searches.[/bold green]")
    return {"search_plan": plan, "error": None}


async def node_research(state: PipelineState) -> dict:
    if state.get("error"):
        return {}
    plan = state["search_plan"]
    assert plan is not None
    _active_console().print(f"[bold green]Executing {len(plan.searches)} search tasks...[/bold green]")
    results = await perform_searches(plan)
    _active_console().print("[bold green]Research complete.[/bold green]")
    return {"search_results": results}


async def node_write(state: PipelineState) -> dict:
    if state.get("error"):
        return {}
    _active_console().print("[magenta]Generating email draft...[/magenta]")
    draft = await write_email_draft(state["user_prompt"], state.get("search_results") or [])
    if draft is None:
        _active_console().print("[bold red]Email drafting failed.[/bold red]")
        return {"draft": None, "error": "draft_failed"}
    _active_console().print("[bold green]Draft created successfully.[/bold green]")
    return {"draft": draft, "error": None}


_builder = StateGraph(PipelineState)
_builder.add_node("plan", node_plan)
_builder.add_node("research", node_research)
_builder.add_node("write", node_write)
_builder.add_edge(START, "plan")
_builder.add_edge("plan", "research")
_builder.add_edge("research", "write")
_builder.add_edge("write", END)
email_marketing_graph = _builder.compile()


### Streaming logic

In [ ]:
async def run_automation_stream(user_prompt: str):
    """Gradio stream: LangGraph (plan → research → write), then SendGrid dispatch."""
    global _trace_console
    buf = io.StringIO()
    capture_console = Console(file=buf, force_terminal=True, width=100)
    _trace_console = capture_console

    def emit() -> str:
        return f"```ansi\n{buf.getvalue()}\n```"

    try:
        capture_console.print("[bold bright_cyan]Starting research flow (LangGraph)...[/bold bright_cyan]")
        yield emit()

        final_state: Optional[PipelineState] = None
        async for state in email_marketing_graph.astream(
            {"user_prompt": user_prompt},
            stream_mode="values",
        ):
            final_state = state
            yield emit()

        if not final_state or final_state.get("error"):
            yield emit()
            return

        draft = final_state.get("draft")
        if draft is None:
            yield emit()
            return

        recipients = _fetch_subscriber_emails_impl()
        capture_console.print("[yellow]Sending to subscribers...[/yellow]")
        yield emit()
        send_status = _send_marketing_email_impl(
            draft.subject, draft.html_body, recipients
        )
        capture_console.print(f"[dim]{send_status}[/dim]")

        if send_status.startswith("Successfully"):
            capture_console.print("[bold bright_yellow]Success: Campaign Dispatched![/bold bright_yellow]")
        else:
            capture_console.print("[bold red]Dispatch failed.[/bold red]")
        yield emit()

    except Exception as e:
        capture_console.print(f"\n[bold red]Critical system error:[/bold red] {str(e)}")
        yield emit()
    finally:
        _trace_console = None


### UI with execution trace

In [ ]:
def launch_ui():
    dark_mode_js = """
    function() {
        document.documentElement.classList.add('dark');
        document.body.classList.add('dark');
    }
    """

    dark_theme = gr.themes.Default(
        primary_hue=gr.themes.colors.blue,
        neutral_hue=gr.themes.colors.slate
    )

    with gr.Blocks(theme=dark_theme, js=dark_mode_js) as demo:
        gr.Markdown("# 📧 Email Marketing Service Agents with LangGraph")
        
        with gr.Row():
            with gr.Column(scale=1):
                prompt_input = gr.Textbox(
                    label="Campaign Objective", 
                    placeholder="e.g., Promote our new eco-friendly sneakers...",
                    lines=5
                )
                launch_btn = gr.Button("Execute Full Flow", variant="primary")
            
            with gr.Column(scale=2):
                trace_output = gr.Markdown(
                    label="Real-time Execution Trace",
                    sanitize_html=False,
                )

        launch_btn.click(
            fn=run_automation_stream,
            inputs=prompt_input,
            outputs=trace_output,
            show_progress="minimal",
        )

    demo.launch(share=True)


if __name__ == "__main__":
    launch_ui()